### Aqui vou deixar as etapas do projeto.

1. Vou deixar eles exatamente como estavam antes de finalizar a etapa, linhas de teste, linhas de debug e acrescentar algumas observações.
2. usar esse arquivo como fonte de estudo.

#### Ingestão da API (extract)

In [ ]:
import requests
from datetime import datetime, timedelta

def extract_cotacao_dolar(dias_atras=30):
    """
    Consome a API do Banco Central e retorna os dados da cotação do dólar dos últimos N dias em um dicionario.
    """
    # Criação das datas inicial e final para jogar dentro da url da API e pegar o periodo em questão.
    data_final = datetime.today()
    data_inicial = data_final - timedelta(days=dias_atras)

    # Definindo o formato da data para que ele sirva na url e a API encontre o periodo selecionado.
    data_inicial_str = data_inicial.strftime("%m-%d-%Y")
    data_final_str = data_final.strftime("%m-%d-%Y")

    url = (f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='{data_inicial_str}'&@dataFinalCotacao='{data_final_str}'&$format=json")

    print("URL usada:", url) # Linha de debug, usada pra que o programa me informe como a url ficou..
    
    response = requests.get(url)
    print("Status code:", response.status_code) # Linha de debug para ver o status da solicitação, se deu certo ou não.
    print("Resposta bruta:", response.text[:500]) # Linha de debug, para ver o texto que a API envia pra gente, sem interpretação. Texto cru.
    response.raise_for_status() # Essa linha verifica se o a requisição deu certo, se sim, ela continua o código. Caso contrário, ele lança um erro e para a execução.

    dados = response.json()
    return dados["value"] # retorna o parte do dicionario que veio da api que nos interessa, onde estão as informações que buscamos.

def salvar_dados(dados_brutos, output_path="data/raw/cotacao_dolar_json"):
    """
    Salva os dados brutos da API para fins de auditoria e reprocessamento.
    """
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(dados_brutos, f, indent=2)


if __name__=="__main__":
    resultado = extract_cotacao_dolar(dias_atras=30)
    print(f"Registros recebidos: {len(resultado)}")
    print(resultado[:2])

#### Transform 

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def transformar_cotacao(spark, dados_brutos):
    """
    Recebe os dados de uma API como lista de dicionarios e retorno um DataFrame limpo, com data convertida e variação percentual calculada.
    """
    df = spark.createDataFrame(dados_brutos)

    df = df.withColumn("data_cotacao", F.to_date(F.col("dataHoraCotacao")))

    window_spec = Window.orderBy("data_cotacao")
    df = df.withColumn("cotacao_venda_anterior", F.lag("cotacaoVenda", 1).over(window_spec))

    variacao = F.round(((df["cotacaoVenda"] - df["cotacao_venda_anterior"])/ df["cotacao_venda_anterior"]*100), 2)

    df = df.withColumn("variacao_percentual", variacao)

    return df



if __name__=="__main__":
    import os
    from pyspark.sql import SparkSession
    from extract import extract_cotacao_dolar
    import sys
    
    os.environ["PYSPARK_PYTHON"] = sys.executable
    os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

    os.environ["HADOOP_HOME"] = "C:\\hadoop"
    os.environ["PATH"] += os.pathsep + "C:\\hadoop\\bin"

    spark = SparkSession.builder.appName("teste-transform").getOrCreate()

    dados = extract_cotacao_dolar(dias_atras=30)
    df= transformar_cotacao(spark, dados)
    df.printSchema()
    df.show(5)

#### Load

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()

def load_cotacao(df_spark, table_name="cotacao_dolar"):
    """
    Recebe um DataFrame Spark, converte para Pandas e escreve na tabela especificada do banco PostgreSQL.

    """
    database_url = os.environ["DATABASE_URL"] # puxando a url de conexão com login e senha de um arquivo .env. Feito dessa forma para garantir segurança

    df_pandas = df_spark.toPandas() # transformando o dataframe spark em dataframe pandas, com pandas é possivel usar o SQL.
    engine = create_engine(database_url) # criando a engine de conexão com a url que chamaos do arquivo .env
    df_pandas.to_sql(table_name, engine, if_exists="replace", index=False)
    # Aqui ele passa o dataframe pandas para uma tabela relacional (sql). Como argumentos temos o nome da tabela, a engine de conexão que criamos para ele saber onde será criada essa tabela,
    # o "if_exists" diz o que fazer caso já exista a tabela no banco, nesse caso, substituir(podendo ser, replace, append ou fail). 
    # replace --> recria a tabela do zero. append --> mantem a tabela como está e adiciona as novas linhas no final. fail --> se a tabela existir n faz nada, lança um erro avisando que ja existe uma tabela.
    # o último argumento é para que não se crie uma coluna de index
    """
    Essa linha diz: "pega esse DataFrame Pandas, e escreve o conteúdo dele numa tabela chamada cotacao_dolar no banco (usando a conexão engine); 
    se essa tabela já existir, apaga e recria do zero; 
    e não inclui a numeração automática de linha do Pandas como coluna"
    """

    print(f"{len(df_pandas)} registros escritos na tabela '{table_name}'.")

if __name__=="__main__":
    import os
    from pyspark.sql import SparkSession
    from extract import extract_cotacao_dolar
    import sys
    from transform import transformar_cotacao
    
    os.environ["PYSPARK_PYTHON"] = sys.executable
    os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

    os.environ["HADOOP_HOME"] = "C:\\hadoop"
    os.environ["PATH"] += os.pathsep + "C:\\hadoop\\bin"

    spark = SparkSession.builder.appName("teste-load").getOrCreate()
    dados = extract_cotacao_dolar(dias_atras=30)
    df= transformar_cotacao(spark, dados)
    load_cotacao(df)